In [ ]:
%pip install google-cloud-pubsub google-auth

  Using cached google_cloud_pubsub-2.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpc_google_iam_v1-0.14.3-py3-none-any.whl.metadata (9.2 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached opentelemetry_semantic_conventions-0.61b0-py3-none-any.whl.metadata (2.5 kB)
Using cached google_cloud_pubsub-2.36.0-py3-none-any.whl (323 kB)
Using cached grpc_google_iam_v1-0.14.3-py3-none-any.whl (32 kB)
Using cached proto_plus-1.27.2-py3-none-any.whl (50 kB)
Using cached opentelemetry_api-1.40.0-py3-none-any.whl (68 kB)
Using cached importlib_metadata-8.7.1-py3-none-any.whl (27 kB)
Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl (141 kB)
Using cached opentelemetr

In [1]:
# 1. Import the Connect tools
from databricks.connect import DatabricksSession
from databricks.sdk.runtime import dbutils # This fixes the 'dbutils is not defined' error

# 2. Wake up the Serverless connection
# This uses your configuration (host/token) to link VS Code to the cloud
spark = DatabricksSession.builder.serverless().getOrCreate()

print("🚀 Successfully connected to Databricks Serverless!")

🚀 Successfully connected to Databricks Serverless!


In [2]:
import os
import tempfile

print("🔐 Fetching secure credentials from Databricks Secrets...")

# 1. Pull the JSON key that Terraform injected into the 'gcp-auth' scope
try:
    gcp_json_key = dbutils.secrets.get(scope="gcp-auth", key="databricks-workspace-sa-json")
    
    # 2. Write it to a temporary secure file on the Serverless node
    temp_key_path = os.path.join(tempfile.gettempdir(), "gcp_key.json")
    with open(temp_key_path, "w") as f:
        f.write(gcp_json_key)

    # 3. Tell the Google SDK to use this specific file for authentication
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = temp_key_path
    print("✅ Identity Loaded into environment. Ready to connect.")

except Exception as e:
    print(f"❌ FAIL: Could not load secret. Did Terraform finish applying? Error: {e}")

🔐 Fetching secure credentials from Databricks Secrets...
✅ Identity Loaded into environment. Ready to connect.


In [5]:
import google.auth
from google.cloud import pubsub_v1

print("🔍 Checking Active GCP Identity...")

# Because of Cell 2, Google will now read our specific Service Account file
credentials, project = google.auth.default()

if hasattr(credentials, 'service_account_email'):
    active_email = credentials.service_account_email
    print(f"👤 Active GCP Identity: {active_email}")
    
    if "databricks-workspace-sa" not in active_email:
        print("⚠️ WARNING: You are not using the dedicated Service Account we built in Terraform.")
else:
    print("⚠️ WARNING: Using User Credentials (ADC), not a Service Account.")


PROJECT_ID = "woolie-project"
SUBSCRIPTION_ID = "bitcoin-price-sub"

print(f"\n📡 Connecting to {SUBSCRIPTION_ID}...")

try:
    subscriber = pubsub_v1.SubscriberClient()
    subscription_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)
    
    # Pull 1 message with a 10s timeout
    response = subscriber.pull(
        request={"subscription": subscription_path, "max_messages": 1},
        timeout=10.0
    )
    
    if response.received_messages:
        print("🟢 PASS: Connection successful. Data received.")
        msg = response.received_messages[0].message
        print(f"📦 Payload: {msg.data.decode('utf-8')}")
    else:
        print("🟡 PASS: Connection successful, but no messages in queue.")
        print("💡 Tip: Is your Cloud Run ingestion job pushing data right now?")
        
except Exception as e:
    print(f"❌ FAIL: Connection could not be established.")
    print(f"Technical Error: {e}")

🔍 Checking Active GCP Identity...
👤 Active GCP Identity: databricks-workspace-sa@woolie-project.iam.gserviceaccount.com

📡 Connecting to bitcoin-price-sub...
🟢 PASS: Connection successful. Data received.
📦 Payload: {"metadata": {"source": "coingecko", "ingested_at": "2026-03-31T00:40:22.228404+00:00", "environment": "dev"}, "data": {"asset": "ethereum", "price_aud": 2960.31, "price_usd": 2026.79, "volume_aud": 25041661575.46076, "volume_usd": 17144906318.93076, "source_timestamp": 1774917613}}
